# Módulo 2.8: Web Scraping con BeautifulSoup y Requests

El **Web Scraping** es una técnica utilizada para extraer grandes cantidades de datos de sitios web. En la ciencia de datos, es una fuente invaluable de información cuando no existen APIs disponibles. Esta sesión se enfocará en dos librerías esenciales de Python para web scraping:
-   **`requests`**: Para hacer solicitudes HTTP y obtener el contenido de las páginas web.
-   **`BeautifulSoup4` (bs4)**: Para parsear el HTML/XML y navegar por la estructura del documento de manera sencilla y eficiente.

## 1. Fundamentos de HTML y el DOM

Antes de hacer scraping, es fundamental entender cómo se estructuran las páginas web:

-   **HTML (HyperText Markup Language):** El lenguaje estándar para crear páginas web. Se compone de elementos (`tags`) anidados que definen la estructura y el contenido (e.g., `<p>`, `<a>`, `<div>`).
-   **DOM (Document Object Model):** Una representación en forma de árbol de la estructura de un documento HTML/XML. BeautifulSoup construye este árbol para que podamos navegarlo y buscar elementos.

## 2. Obtener Contenido Web con `requests` y Parsear con `BeautifulSoup`

Primero, usaremos `requests` para descargar el HTML de una página. Luego, pasaremos el texto HTML a `BeautifulSoup` para que lo parsee y nos proporcione un objeto manipulable.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL del sitio web a scrapear
url_nostarch = 'https://nostarch.com'

print(f"Obteniendo contenido de: {url_nostarch}")
try:
    # Realizar la solicitud HTTP GET
    response = requests.get(url_nostarch)
    response.raise_for_status() # Lanza una excepción si la solicitud HTTP falla

    # Parsear el contenido HTML con BeautifulSoup
    # 'lxml' es un parser rápido, 'html.parser' es el parser estándar de Python
    soup = BeautifulSoup(response.text, 'lxml')
    print("Contenido HTML parseado exitosamente.")

except requests.exceptions.RequestException as e:
    print(f"Error al obtener la página web: {e}")
    soup = None # En caso de error, el objeto soup será None


### 2.1. Navegación Básica del Árbol HTML

BeautifulSoup nos permite acceder a elementos HTML como si fueran atributos del objeto `soup` o usar métodos como `.find()` y `.find_all()`.

In [ ]:
if soup:
    # Acceder al título de la página
    titulo = soup.title
    if titulo:
        print(f"Título de la página (tag): {titulo}")
        print(f"Nombre del tag: {titulo.name}")
        print(f"Contenido del tag: {titulo.string}")
        print(f"Padre del tag 'title': {titulo.parent.name}")
    else:
        print("No se encontró el tag <title>.")

    # Acceder al primer párrafo
    primer_parrafo = soup.p
    if primer_parrafo:
        print(f"\nPrimer párrafo (tag <p>): {primer_parrafo}")
        print(f"Texto del primer párrafo: {primer_parrafo.get_text()}")
    else:
        print("No se encontró ningún tag <p>.")

## 3. Búsqueda de Elementos con `find()` y `find_all()`

Estos métodos son fundamentales para localizar elementos específicos o colecciones de elementos dentro del árbol HTML.

In [ ]:
if soup:
    # Encontrar todos los enlaces (tags <a>)
    todos_los_enlaces = soup.find_all('a')
    print(f"Número total de enlaces: {len(todos_los_enlaces)}")

    # Encontrar todos los párrafos (tags <p>)
    todos_los_parrafos = soup.find_all('p')
    print(f"Número total de párrafos: {len(todos_los_parrafos)}")

    # Encontrar un elemento por su ID (si el elemento tiene un atributo id)
    # Ejemplo hipotético: elemento_por_id = soup.find(id='main-content')
    
    # Encontrar elementos por clase CSS (ejemplo específico de No Starch Press)
    # Nota: La clase 'product-body' puede variar o no existir en el sitio web actual.
    print("\nBuscando elementos con clase 'product-body':")
    libros_info = soup.find_all('div', class_='product-body')
    print(f"Elementos 'product-body' encontrados: {len(libros_info)}")

    if libros_info:
        primer_libro = libros_info[0]
        titulo_libro = primer_libro.a.get_text(strip=True) if primer_libro.a else 'N/A'
        url_libro = primer_libro.a['href'] if primer_libro.a and 'href' in primer_libro.a.attrs else 'N/A'
        print(f"Primer libro encontrado: Título='{titulo_libro}', URL='{url_libro}'")

## 4. Selectores CSS con `.select()`

BeautifulSoup también soporta la selección de elementos utilizando selectores CSS, a través del método `.select()`. Esto puede ser muy potente para localizar elementos basándose en su jerarquía, clases, IDs o atributos.

In [ ]:
if soup:
    # Seleccionar todos los enlaces (<a> tags) dentro de un párrafo (<p>)
    enlaces_en_parrafos = soup.select('p a')
    print(f"Enlaces encontrados en párrafos: {len(enlaces_en_parrafos)}")
    if enlaces_en_parrafos:
        print(f"Primer enlace en párrafo: {enlaces_en_parrafos[0].get_text(strip=True)} -> {enlaces_en_parrafos[0]['href']}")

    # Seleccionar un elemento por su ID
    # Ejemplo: footer_element = soup.select('#footer')

    # Seleccionar elementos con una clase específica
    # Ejemplo: all_product_bodies = soup.select('.product-body')

    # Ejemplo más complejo: Enlaces que contienen un dominio específico en su atributo 'href'
    # Nota: Este selector puede ser muy específico y puede que no encuentre nada dependiendo de la página.
    enlaces_externos = soup.select('a[href*="http"]') # Enlaces cuyo href contenga 'http'
    print(f"\nEnlaces externos encontrados: {len(enlaces_externos)}")
    if enlaces_externos:
        print(f"Algunos enlaces externos:\n")
        for i, link in enumerate(enlaces_externos[:5]): # Mostrar los primeros 5
            print(f"  - {link.get_text(strip=True)[:50]}... -> {link['href']}")

## 5. Ejercicio Práctico: Extracción Estructurada a DataFrame

Vamos a extraer títulos y URLs de los libros de No Starch Press y organizarlos en un DataFrame de Pandas.

In [ ]:
if soup:
    libros_data = []
    # La clase 'product-body' puede variar o no existir.
    # Se asume una estructura donde el título y URL están dentro de un <a> dentro de 'product-body'.
    for libro_div in soup.select('div.product-body'):
        titulo_tag = libro_div.find('a')
        if titulo_tag:
            titulo = titulo_tag.get_text(strip=True)
            url = titulo_tag['href']
            libros_data.append({'Titulo': titulo, 'URL': url})

    if libros_data:
        df_libros = pd.DataFrame(libros_data)
        print("DataFrame de libros de No Starch Press:\n")
        print(df_libros.head())
        # Opcional: Guardar a CSV
        # df_libros.to_csv('data/nostarch_libros.csv', index=False)
        # print("\nDatos guardados en 'data/nostarch_libros.csv'")
    else:
        print("No se encontraron datos de libros con los selectores especificados.")
else:
    print("No hay objeto BeautifulSoup para procesar. Asegúrate de que la descarga de la URL fue exitosa.")

## 6. Consideraciones Éticas y Legales del Web Scraping

El web scraping, aunque potente, debe realizarse con responsabilidad y ética:

-   **`robots.txt`:** Consulta siempre el archivo `robots.txt` del sitio web (e.g., `https://nostarch.com/robots.txt`). Este archivo especifica qué partes del sitio no deben ser rastreadas o scrapeadas por bots.
-   **Términos de Servicio:** Revisa los términos de servicio del sitio. Algunos prohíben explícitamente el scraping.
-   **Frecuencia:** No realices solicitudes excesivas en un corto período de tiempo para evitar sobrecargar los servidores del sitio web. Utiliza `time.sleep()` entre solicitudes.
-   **Uso de los Datos:** Respeta la privacidad y los derechos de autor. No uses los datos scrapeados para fines maliciosos o ilegales.